# Velora Remote CatVTON Worker Colab Bootstrap

**Research-only and non-commercial.** This notebook is for isolated virtual try-on experiments only. Do not upload production secrets, private user images, customer data, model weights without approval, or generated outputs intended for product use. Confirm CatVTON license, model weight access, and dataset restrictions before running any inference.

Run top-to-bottom only after selecting a T4 GPU runtime.

In [ ]:
# 2. GPU, Python, disk, and CUDA verification
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Disk:", shutil.disk_usage('/content'))
subprocess.run(["nvidia-smi"], check=True)


In [ ]:
# 3. Configuration variables
from pathlib import Path

VELORA_REPO_URL = "<VELORA_REPOSITORY_URL>"
CATVTON_REPO_URL = "<CATVTON_REPOSITORY_URL>"
VELORA_BRANCH = "main"
CATVTON_BRANCH = "main"

PRODUCT_ID = "<PRODUCT_ID_FROM_LOCAL_BACKEND>"
PERSON_IMAGE_ASSET_ID = "person-smoke.png"

EXECUTOR_MODE = "catvton-research"  # Explicit real mode. Use "fake" only for worker API checks.
CLOTH_TYPE = "upper"
SEED = 42
INFERENCE_STEPS = 30
GUIDANCE_SCALE = 2.5
WIDTH = 768
HEIGHT = 1024
DEVICE = "cuda"

BASE_MODEL_PATH = "runwayml/stable-diffusion-inpainting"
RESUME_PATH = "zhengchong/CatVTON"

WORKSPACE = Path("/content")
VELORA_ROOT = WORKSPACE / "velora"
CATVTON_ROOT = WORKSPACE / "CatVTON"
RESEARCH_INPUT_ROOT = WORKSPACE / "velora-research-inputs"
PERSON_ROOT = RESEARCH_INPUT_ROOT / "person"
CATALOG_GARMENT_ROOT = RESEARCH_INPUT_ROOT / "catalog-garments"
WARDROBE_GARMENT_ROOT = RESEARCH_INPUT_ROOT / "wardrobe-garments"
WORKER_LOG_PATH = WORKSPACE / "velora-ai-worker.log"
TUNNEL_LOG_PATH = WORKSPACE / "velora-cloudflared.log"

print("Velora root:", VELORA_ROOT)
print("CatVTON root:", CATVTON_ROOT)


In [ ]:
# 4. Clone or update Velora and CatVTON
import subprocess

def clone_or_update(repo_url: str, target: Path, branch: str) -> None:
    if target.exists():
        subprocess.run(["git", "-C", str(target), "fetch", "--all", "--prune"], check=True)
        subprocess.run(["git", "-C", str(target), "checkout", branch], check=True)
        subprocess.run(["git", "-C", str(target), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", branch, repo_url, str(target)], check=True)

clone_or_update(VELORA_REPO_URL, VELORA_ROOT, VELORA_BRANCH)
clone_or_update(CATVTON_REPO_URL, CATVTON_ROOT, CATVTON_BRANCH)


In [ ]:
# 5. Install the verified CUDA-compatible CatVTON environment
# This cell intentionally installs CUDA Torch 2.4.0 cu121 first.
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade", "pip",
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "torch==2.4.0", "torchvision==0.19.0", "torchaudio==2.4.0",
    "--index-url", "https://download.pytorch.org/whl/cu121",
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "diffusers==0.30.3",
    "accelerate==0.34.2",
    "transformers==4.46.3",
    "huggingface-hub==0.36.2",
    "fvcore==0.1.5.post20221221",
    "opencv-python==4.10.0.84",
    "gradio==4.41.0",
], check=True)

# Install CatVTON local requirements if present, without changing the pinned CUDA Torch package.
requirements = CATVTON_ROOT / "requirements.txt"
if requirements.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-r", str(requirements)], check=False)


## 6. Restart-runtime checkpoint

If Colab requests a restart after dependency installation, restart now:

`Runtime -> Restart runtime`

After restart, rerun cells 2, 3, 4 if needed, then continue from the post-restart verification cell below. Do not rerun package installation unless versions drift.

In [ ]:
# 7. Post-restart environment verification
import importlib
import subprocess
import sys

import torch

print("torch:", torch.__version__)
print("torch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available() or torch.version.cuda is None:
    raise RuntimeError("CUDA Torch is required. Re-select a GPU runtime and reinstall cu121 Torch.")
print("GPU:", torch.cuda.get_device_name(0))

for package_name in ["diffusers", "accelerate", "transformers", "huggingface_hub", "fvcore", "cv2"]:
    module = importlib.import_module(package_name)
    print(package_name, getattr(module, "__version__", "version unavailable"))

subprocess.run(["nvidia-smi"], check=True)


In [ ]:
# 8. Install AI worker dependencies without replacing CUDA Torch with CPU Torch
# Do not run `uv sync` in the shared Colab runtime because dependency resolution may replace CUDA Torch.
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "fastapi>=0.116.0",
    "uvicorn[standard]>=0.35.0",
    "pydantic>=2.0.0",
], check=True)

import torch
print("CUDA still available after worker deps:", torch.cuda.is_available(), torch.version.cuda)
if not torch.cuda.is_available() or torch.version.cuda is None:
    raise RuntimeError("Worker dependency installation broke CUDA Torch.")


In [ ]:
# 9. Create research input directories
import shutil

for directory in [PERSON_ROOT, CATALOG_GARMENT_ROOT, WARDROBE_GARMENT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)
    for existing_file in directory.glob("*"):
        if existing_file.is_file():
            existing_file.unlink()
print("Prepared research input directories under", RESEARCH_INPUT_ROOT)


In [ ]:
# 10. Copy the approved CatVTON example person and garment images
from pathlib import Path
import shutil

def first_image(root: Path, keywords: tuple[str, ...]) -> Path:
    candidates = []
    for pattern in ["*.png", "*.jpg", "*.jpeg", "*.webp"]:
        candidates.extend(root.rglob(pattern))
    keyword_matches = [path for path in candidates if any(keyword in str(path).lower() for keyword in keywords)]
    selected = (keyword_matches or candidates)
    if not selected:
        raise FileNotFoundError(f"No example images found under {root}")
    return sorted(selected)[0]

APPROVED_PERSON_EXAMPLE = first_image(CATVTON_ROOT, ("person", "model", "human"))
APPROVED_GARMENT_EXAMPLE = first_image(CATVTON_ROOT, ("cloth", "garment", "upper", "shirt"))

person_target = PERSON_ROOT / PERSON_IMAGE_ASSET_ID
garment_target = CATALOG_GARMENT_ROOT / f"{PRODUCT_ID}.png"
shutil.copyfile(APPROVED_PERSON_EXAMPLE, person_target)
shutil.copyfile(APPROVED_GARMENT_EXAMPLE, garment_target)

print("Person example:", APPROVED_PERSON_EXAMPLE, "->", person_target)
print("Garment example:", APPROVED_GARMENT_EXAMPLE, "->", garment_target)


In [ ]:
# 11. Configure all AI worker environment variables
import os

os.environ["VELORA_AI_WORKER_EXECUTOR_MODE"] = EXECUTOR_MODE
os.environ["VELORA_ML_PATH"] = str(VELORA_ROOT / "ml")
os.environ["CATVTON_SOURCE_PATH"] = str(CATVTON_ROOT)
os.environ["CATVTON_PERSON_IMAGE_ROOT"] = str(PERSON_ROOT)
os.environ["CATVTON_CATALOG_GARMENT_ROOT"] = str(CATALOG_GARMENT_ROOT)
os.environ["CATVTON_WARDROBE_GARMENT_ROOT"] = str(WARDROBE_GARMENT_ROOT)
os.environ["CATVTON_PERSON_IMAGE_PATH_TEMPLATE"] = "{personImageAssetId}"
os.environ["CATVTON_CATALOG_GARMENT_PATH_TEMPLATE"] = "{productId}.png"
os.environ["CATVTON_WARDROBE_GARMENT_PATH_TEMPLATE"] = "{wardrobeItemId}.png"
os.environ["CATVTON_BASE_MODEL_PATH"] = BASE_MODEL_PATH
os.environ["CATVTON_RESUME_PATH"] = RESUME_PATH
os.environ["CATVTON_DEVICE"] = DEVICE
os.environ["CATVTON_CLOTH_TYPE"] = CLOTH_TYPE
os.environ["CATVTON_SEED"] = str(SEED)
os.environ["CATVTON_INFERENCE_STEPS"] = str(INFERENCE_STEPS)
os.environ["CATVTON_GUIDANCE_SCALE"] = str(GUIDANCE_SCALE)
os.environ["CATVTON_WIDTH"] = str(WIDTH)
os.environ["CATVTON_HEIGHT"] = str(HEIGHT)
os.environ["CATVTON_TIMEOUT_SECONDS"] = "1200"

for name in sorted(key for key in os.environ if key.startswith(("VELORA_", "CATVTON_"))):
    print(name, "=", os.environ[name])


In [ ]:
# 12. Start the AI worker in the background
import os
import signal
import subprocess
import time

def stop_process_file(pid_path: Path) -> None:
    if not pid_path.exists():
        return
    try:
        pid = int(pid_path.read_text().strip())
        os.kill(pid, signal.SIGTERM)
        time.sleep(2)
    except Exception as error:
        print("Previous process stop skipped:", error)
    finally:
        pid_path.unlink(missing_ok=True)

WORKER_PID_PATH = WORKSPACE / "velora-ai-worker.pid"
stop_process_file(WORKER_PID_PATH)
WORKER_LOG_PATH.unlink(missing_ok=True)

worker_log = WORKER_LOG_PATH.open("w", encoding="utf-8")
worker_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "src.app:app", "--host", "0.0.0.0", "--port", "4100"],
    cwd=VELORA_ROOT / "ai-worker",
    stdout=worker_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)
WORKER_PID_PATH.write_text(str(worker_process.pid), encoding="utf-8")
print("Started worker PID", worker_process.pid)
time.sleep(5)


In [ ]:
# 13. Verify local /health
import json
import urllib.request

with urllib.request.urlopen("http://127.0.0.1:4100/health", timeout=10) as response:
    body = response.read().decode("utf-8")
    print(response.status, body)
    if response.status != 200:
        raise RuntimeError("Worker health failed")


In [ ]:
# 14. Download and start Cloudflare Quick Tunnel
import os
import re
import signal
import subprocess
import time
import urllib.request

CLOUDFLARED_PATH = WORKSPACE / "cloudflared"
TUNNEL_PID_PATH = WORKSPACE / "velora-cloudflared.pid"
stop_process_file(TUNNEL_PID_PATH)
TUNNEL_LOG_PATH.unlink(missing_ok=True)

if not CLOUDFLARED_PATH.exists():
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(url, CLOUDFLARED_PATH)
    CLOUDFLARED_PATH.chmod(0o755)

tunnel_log = TUNNEL_LOG_PATH.open("w", encoding="utf-8")
tunnel_process = subprocess.Popen(
    [str(CLOUDFLARED_PATH), "tunnel", "--url", "http://127.0.0.1:4100", "--no-autoupdate"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)
TUNNEL_PID_PATH.write_text(str(tunnel_process.pid), encoding="utf-8")
print("Started cloudflared PID", tunnel_process.pid)
time.sleep(8)


In [ ]:
# 15. Extract and display the temporary public worker URL
import re
import time

PUBLIC_WORKER_URL = None
for _ in range(30):
    text = TUNNEL_LOG_PATH.read_text(encoding="utf-8", errors="ignore") if TUNNEL_LOG_PATH.exists() else ""
    match = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", text)
    if match:
        PUBLIC_WORKER_URL = match.group(0)
        break
    time.sleep(2)

if PUBLIC_WORKER_URL is None:
    raise RuntimeError("Could not extract Cloudflare tunnel URL. Check tunnel logs.")
print("PUBLIC_WORKER_URL=", PUBLIC_WORKER_URL)


In [ ]:
# 16. Verify public /health
with urllib.request.urlopen(f"{PUBLIC_WORKER_URL}/health", timeout=20) as response:
    body = response.read().decode("utf-8")
    print(response.status, body)
    if response.status != 200:
        raise RuntimeError("Public worker health failed")


In [ ]:
# 17. Print exact local PowerShell environment commands using the generated public URL
powershell = f'''
$env:TRY_ON_EXECUTOR_MODE="remote-http"
$env:TRY_ON_REMOTE_WORKER_BASE_URL="{PUBLIC_WORKER_URL}"
$env:TRY_ON_REMOTE_WORKER_SUBMIT_PATH="/jobs"
$env:TRY_ON_REMOTE_WORKER_STATUS_PATH="/jobs/{{workerJobId}}"
$env:TRY_ON_REMOTE_WORKER_CANCEL_PATH="/jobs/{{workerJobId}}/cancel"
$env:TRY_ON_REMOTE_WORKER_RESULT_PATH="/jobs/{{workerJobId}}/result"
$env:TRY_ON_REMOTE_WORKER_TIMEOUT_MS="600000"
$env:TRY_ON_REMOTE_WORKER_POLL_INTERVAL_MS="5000"
$env:TRY_ON_REMOTE_WORKER_MAX_WAIT_MS="1200000"
$env:TRY_ON_SMOKE_BACKEND_BASE_URL="http://127.0.0.1:4000/api/v1"
$env:TRY_ON_SMOKE_WORKER_BASE_URL="{PUBLIC_WORKER_URL}"
$env:TRY_ON_SMOKE_AUTH_TOKEN="<LOCAL_TEST_USER_ACCESS_TOKEN>"
$env:TRY_ON_SMOKE_PRODUCT_ID="{PRODUCT_ID}"
$env:TRY_ON_SMOKE_PERSON_IMAGE_PATH="<LOCAL_PERSON_IMAGE_PATH>"
$env:TRY_ON_SMOKE_GARMENT_IMAGE_PATH="<LOCAL_GARMENT_IMAGE_PATH>"
$env:TRY_ON_SMOKE_PERSON_IMAGE_ASSET_ID="{PERSON_IMAGE_ASSET_ID}"
$env:TRY_ON_SMOKE_TIMEOUT_MS="1200000"
$env:TRY_ON_SMOKE_POLL_INTERVAL_MS="5000"
cd C:\\projects\\velora\\backend
npm run tryon:remote-smoke -- --dry-run
npm run tryon:remote-smoke
'''
print(powershell)


In [ ]:
# 18. Show worker logs
print("--- worker log ---")
print(WORKER_LOG_PATH.read_text(encoding="utf-8", errors="ignore")[-8000:] if WORKER_LOG_PATH.exists() else "missing worker log")
print("--- tunnel log ---")
print(TUNNEL_LOG_PATH.read_text(encoding="utf-8", errors="ignore")[-8000:] if TUNNEL_LOG_PATH.exists() else "missing tunnel log")


In [ ]:
# 19. Stop worker/tunnel and clean research files
stop_process_file(WORKER_PID_PATH)
stop_process_file(TUNNEL_PID_PATH)

for path in [RESEARCH_INPUT_ROOT, VELORA_ROOT / "ai-worker" / "data" / "output", VELORA_ROOT / "ml" / "data" / "output"]:
    if path.exists():
        for child in path.glob("**/*"):
            if child.is_file() and child.name != ".gitkeep":
                child.unlink()
print("Stopped worker/tunnel and cleaned research inputs/generated outputs.")
